#### Noisy QSVM (SVC Precomputed Kernel) - Spambase - Consistent with Ideal

In [1]:
%pip install qiskit qiskit-machine-learning qiskit-aer seaborn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import qiskit, qiskit_aer, qiskit_machine_learning
print("Qiskit:", qiskit.__version__)
print("Aer:", qiskit_aer.__version__)
print("QML:", qiskit_machine_learning.__version__)

Qiskit: 2.2.3
Aer: 0.17.2
QML: 0.9.0


In [3]:
# To ensure reproducibility of results
from qiskit_machine_learning.utils import algorithm_globals
algorithm_globals.random_seed = 12345

In [4]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import time
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, recall_score, balanced_accuracy_score

In [5]:
# --- Qiskit Imports ---
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

##### Load Dataset

In [6]:
# --- Import Spambase Column Names ---
spambase_columns = [
    "word_freq_make", "word_freq_address", "word_freq_all", "word_freq_3d", "word_freq_our",
    "word_freq_over", "word_freq_remove", "word_freq_internet", "word_freq_order", "word_freq_mail",
    "word_freq_receive", "word_freq_will", "word_freq_people", "word_freq_report", "word_freq_addresses",
    "word_freq_free", "word_freq_business", "word_freq_email", "word_freq_you", "word_freq_credit",
    "word_freq_your", "word_freq_font", "word_freq_000", "word_freq_money", "word_freq_hp",
    "word_freq_hpl", "word_freq_george", "word_freq_650", "word_freq_lab", "word_freq_labs",
    "word_freq_telnet", "word_freq_857", "word_freq_data", "word_freq_415", "word_freq_85",
    "word_freq_technology", "word_freq_1999", "word_freq_parts", "word_freq_pm", "word_freq_direct",
    "word_freq_cs", "word_freq_meeting", "word_freq_original", "word_freq_project", "word_freq_re",
    "word_freq_edu", "word_freq_table", "word_freq_conference", "char_freq_;", "char_freq_(",
    "char_freq_[", "char_freq_!", "char_freq_$", "char_freq_#", "capital_run_length_average",
    "capital_run_length_longest", "capital_run_length_total", "label"
]

# --- 1. Load the Spambase Dataset (LOCAL PATH) ---
# file_path = '/kaggle/input/spambase/spambase.data'
file_path = r'data/spambase/spambase.data'
df = pd.read_csv(file_path, header=None, names=spambase_columns)
df.drop_duplicates(inplace=True)

print(f"Dataset loaded: {df.shape[0]} samples, {df.shape[1]} features")

Dataset loaded: 4210 samples, 58 features


##### Noise Model Factory Function

In [7]:
def get_noise_model(level='standard'):
    """
    Returns a noise model, backend, and pass manager for the given noise level.
    
    Noise Levels:
    - 'low':      0.01% 1q, 0.1% 2q, 0.2% readout
    - 'standard': 0.1% 1q, 1.0% 2q, 2.0% readout (realistic NISQ device)
    - 'high':     0.5% 1q, 5.0% 2q, 10.0% readout
    """
    noise_params = {
        'low':      {'p_1q': 0.0001, 'p_2q': 0.001,  'p_ro': 0.002},
        'standard': {'p_1q': 0.001,  'p_2q': 0.01,   'p_ro': 0.02},
        'high':     {'p_1q': 0.005,  'p_2q': 0.05,   'p_ro': 0.10},
    }
    
    params = noise_params.get(level, noise_params['standard'])
    
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(params['p_1q'], 1), ['u1', 'u2', 'u3'])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(params['p_2q'], 2), ['cx'])
    readout_error = ReadoutError([[1 - params['p_ro'], params['p_ro']], [params['p_ro'], 1 - params['p_ro']]])
    noise_model.add_all_qubit_readout_error(readout_error)
    
    backend = AerSimulator(noise_model=noise_model, seed_simulator=12345)
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    
    return noise_model, backend, pm, params

print("Noise model factory function ready!")
print("Available levels: 'low', 'standard', 'high'")

Noise model factory function ready!
Available levels: 'low', 'standard', 'high'


##### Experiment Configurations

In [8]:
# ==========================================
# EXPERIMENT CONFIGURATIONS
# ==========================================
# All experiments use 'noise_level' key. If not specified, defaults to 'standard'.

experiments = [
    # --- EXP 1: Sample Size Effect (Generalization) ---
    {'id': 'Exp1_200samp',  'samples': 200, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp1_400samp',  'samples': 400, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp1_600samp',  'samples': 600, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp1_800samp',  'samples': 800, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp1_1000samp',  'samples': 1000, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 2: Dimensionality Effect (Quantum Complexity) ---
    {'id': 'Exp2_2feat',   'samples': 300, 'k_features': 2,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_4feat',   'samples': 300, 'k_features': 4,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_6feat',  'samples': 300, 'k_features': 6, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_8feat',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_10feat',  'samples': 300, 'k_features': 10, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_12feat',  'samples': 300, 'k_features': 12, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 3: Shot Noise Effect (Measurement Precision) ---
    {'id': 'Exp3_128shots',  'samples': 300, 'k_features': 8, 'shots': 128,  'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp3_512shots',  'samples': 300, 'k_features': 8, 'shots': 512,  'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp3_1024shots', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 4: Reps Effect (Circuit Complexity) ---
    {'id': 'Exp4_Reps1', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1,  'entanglement': 'linear', 'noise_level': 'standard'},  
    {'id': 'Exp4_Reps2', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 2,  'entanglement': 'linear', 'noise_level': 'standard'},  
    {'id': 'Exp4_Reps3', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 3,  'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 5: Entanglement Ablation ---
    {'id': 'Exp5_Linear',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp5_Circular', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'circular', 'noise_level': 'standard'},
    {'id': 'Exp5_Full',    'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'full', 'noise_level': 'standard'},

    # --- EXP 6: Noise Ablation (Fixed quantum params, varying noise levels) ---
    {'id': 'Exp6_LowNoise',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'low'},
    {'id': 'Exp6_StdNoise',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp6_HighNoise', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'high'},
]

print(f"Total experiments configured: {len(experiments)}")
print("Running might take hours or even several days (Noisy Simulation)")

Total experiments configured: 21
Running might take hours or even several days (Noisy Simulation)


##### Main Experiment Loop

In [ ]:
for i, config in enumerate(experiments, 1):
    print("="*80)
    print(f"EXPERIMENT {i}/{len(experiments)}: {config['id']}")
    print("="*80)
    print(f"  Samples: {config['samples']}")
    print(f"  K Features: {config['k_features']}")
    print(f"  Shots: {config['shots']}")
    print(f"  Reps: {config['reps']}")
    print(f"  Entanglement: {config['entanglement']}")
    print(f"  Noise Level: {config.get('noise_level', 'standard')}")
    print("="*80)
    
    # --- 0. Get Noise Model for this experiment ---
    noise_level = config.get('noise_level', 'standard')
    noise_model, noisy_backend, pm, noise_params = get_noise_model(noise_level)
    print(f"Noise Model: 1q={noise_params['p_1q']*100:.2f}%, 2q={noise_params['p_2q']*100:.2f}%, readout={noise_params['p_ro']*100:.2f}%")
    
    # --- 1. Data Preparation ---
    X = df.drop('label', axis=1)
    y = df['label']
    
    # Split train/test (Keep stratified)
    X_subset, _, y_subset, _ = train_test_split(
        X, y,
        train_size=config['samples'], # Dynamic sample size
        stratify=y,
        random_state=42
    )
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset,
        test_size=0.30,
        random_state=42,
        stratify=y_subset
    )
    
    print(f"\nDataset created: {X_train.shape[0]} train, {X_test.shape[0]} test")
    
    # Standard Scaling
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)
    print("Data scaled successfully")
    
    # Drop Highly Correlated Features (Correlation Analysis)
    corr_matrix = X_train_scaled.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
    
    X_train_scaled.drop(columns=to_drop, inplace=True)
    X_test_scaled.drop(columns=to_drop, inplace=True)
    print(f"Dropped {len(to_drop)} highly correlated features")
    
    # SelectKBest
    k = config['k_features']
    selector = SelectKBest(score_func=f_classif, k=k)
    X_train_kbest = selector.fit_transform(X_train_scaled, y_train)
    X_test_kbest = selector.transform(X_test_scaled)
    
    # Get selected feature names
    cols = X_train_scaled.columns
    selected_indices = selector.get_support(indices=True)
    selected_features = cols[selected_indices]
    print(f"SelectKBest: Selected {k} features:")
    for idx, name in enumerate(selected_features, 1):
        print(f"     {idx}. {name}")
        
    # --- 2. Quantum Kernel Setup (NOISY) ---
    # Noisy Sampler for this experiment (configure shots dynamically)
    noise_sampler = AerSampler.from_backend(
        backend=noisy_backend,
        default_shots=config['shots']
    )
    
    feature_map = ZZFeatureMap(
        feature_dimension=k, 
        reps=config['reps'], 
        entanglement=config['entanglement']
    )
    
    # Fidelity with NOISY sampler
    fidelity = ComputeUncompute(sampler=noise_sampler, pass_manager=pm)
    
    # Quantum Kernel
    qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)
    
    print(f"Quantum kernel configured (ZZFeatureMap, reps={config['reps']}, entanglement={config['entanglement']}, shots={config['shots']})")
    
    # --- 3. Compute Kernel Matrices ---
    print("\nComputing kernel matrices...")
    start_k = time.time()
    kernel_train = qkernel.evaluate(x_vec=X_train_kbest)
    kernel_test = qkernel.evaluate(x_vec=X_test_kbest, y_vec=X_train_kbest)
    duration_k = time.time() - start_k
    print(f"Kernel computation: {duration_k:.2f}s")
    
    # --- 4. Train SVC (Precomputed) ---
    print("\nGrid searching for optimal C...")
    param_grid = {'C': [0.1, 1, 10, 100]}
    svc = SVC(kernel='precomputed', class_weight='balanced')
    grid = GridSearchCV(svc, param_grid, cv=3, scoring='accuracy')
    
    start_t = time.time()
    grid.fit(kernel_train, y_train)
    best_model = grid.best_estimator_
    duration_t = time.time() - start_t
    
    # --- 5. Evaluation ---
    print(f"  → Best C: {grid.best_params_['C']}")
    print(f"  → CV Score: {grid.best_score_:.4f}")
    print(f"  → Training time: {duration_t:.2f}s")
    
    y_train_pred = best_model.predict(kernel_train)
    y_test_pred = best_model.predict(kernel_test)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    bal_acc = balanced_accuracy_score(y_test, y_test_pred)
    recall = recall_score(y_test, y_test_pred, pos_label=1)
    gen_gap = abs(train_acc - test_acc)
    
    print(f"  → Train Accuracy: {train_acc:.4f}")
    print(f"  → Test Accuracy: {test_acc:.4f}")
    print(f"  → Test Balanced Accuracy: {bal_acc:.4f}")
    print(f"  → Spam Recall: {recall:.4f}")
    print(f"  → Generalization Gap: {gen_gap:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_test_pred))
    
    # --- 6. Save Matrices (Optional but good for analysis) ---
    # Naming convention: kernel_train_ExpID.npy
    np.save(f'kernel_train_{config["id"]}.npy', kernel_train)
    np.save(f'kernel_test_{config["id"]}.npy', kernel_test)
    print(f"Saved kernel matrices: kernel_train_{config['id']}.npy, kernel_test_{config['id']}.npy")
    print("\n")

EXPERIMENT 1/21: Exp1_500samp
  Samples: 500
  K Features: 8
  Shots: 1024
  Reps: 1
  Entanglement: linear
  Noise Level: standard
Noise Model: 1q=0.10%, 2q=1.00%, readout=2.00%

Dataset created: 350 train, 150 test
Data scaled successfully
Dropped 1 highly correlated features
SelectKBest: Selected 8 features:
     1. word_freq_remove
     2. word_freq_free
     3. word_freq_business
     4. word_freq_your
     5. word_freq_000
     6. char_freq_!
     7. char_freq_$
     8. capital_run_length_longest
Quantum kernel configured (ZZFeatureMap, reps=1, entanglement=linear, shots=1024)

Computing kernel matrices...


/tmp/ipykernel_10040/3381458879.py:75: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(
